<a href="https://colab.research.google.com/github/amadisamantha-arch/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/amadisamantha-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

Loaded 30000 rows


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Paper finding 1: ML Appendix "What Predicts Health?" — Random Forest shows
Average Position (43%), Impressions (32%), Scroll Depth (15%) as top
predictors of Health Score.

Methodology question: Health Score is explicitly defined as Impressions
(30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts) — so three
of the "top predictor" features are literally ingredients of the label
itself. The paper acknowledges this ("importance is descriptive rather
than causal"), but I'd ask: would removing Position and Impressions from
the feature set reveal which signals actually drive Health Score beyond
its own formula? Right now this reads closer to confirming the scoring
formula than discovering new signal.

Paper finding 2: The Freshness Multiplier — 361+ day content shows a
283:1 growth-to-decline ratio, presented as a headline number.

Methodology question: that ratio comes from 283 growing pages vs. only 1
declining page. The paper itself calls this "unstable," but I'd ask: how
sensitive is 283:1 to that single denominator? If 2-3 more declining pages
existed in that bucket, the ratio could fall by an order of magnitude. The
paper sets a minimum bucket size of n=50 elsewhere — why does a bucket
this small still get headline visual treatment (a large "361+  283" stat)
rather than being demoted the way other unstable cuts were?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before/after: my Week-5 model (w05_model.ipynb) already used a client-holdout
split, but I never compared it against a naive row-level random split to show
what an honest split actually protects against. Here I rebuild both splits
side by side to make the leakage risk visible, not just assumed.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

features = ['impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update',
            'avg_position', 'ctr', 'word_count']
features = [f for f in features if f in df.columns]

X_all = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_all = (df['trend_direction'] == 'down').astype(int)

# BEFORE: naive random row-level split (ignores that pages share a client)
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42, stratify=y_all)

model_naive = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
model_naive.fit(X_tr_naive, y_tr_naive)
naive_scores = model_naive.predict_proba(X_te_naive)[:, 1]

# AFTER: honest client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

X_tr_honest = train_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_tr_honest = (train_df['trend_direction'] == 'down').astype(int)
X_te_honest = test_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_te_honest = (test_df['trend_direction'] == 'down').astype(int)

model_honest = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
model_honest.fit(X_tr_honest, y_tr_honest)
honest_scores = model_honest.predict_proba(X_te_honest)[:, 1]

client_overlap = len(set(train_df['client_id']) & set(test_df['client_id']))

comparison = pd.DataFrame({
    'split': ['naive random row-level', 'honest client-holdout'],
    'Precision@20': [
        precision_at_k(naive_scores, y_te_naive, 20),
        precision_at_k(honest_scores, y_te_honest, 20)
    ],
    'Precision@50': [
        precision_at_k(naive_scores, y_te_naive, 50),
        precision_at_k(honest_scores, y_te_honest, 50)
    ],
})
print(comparison)
print(f"\nClient overlap in honest split: {client_overlap} (should be 0)")

                    split  Precision@20  Precision@50
0  naive random row-level           1.0          0.92
1   honest client-holdout           0.6          0.66

Client overlap in honest split: 0 (should be 0)


Result: the naive random row-level split shows Precision@20 = 1.00 and
Precision@50 = 0.92 — suspiciously close to perfect. The honest client-holdout
split shows Precision@20 = 0.60 and Precision@50 = 0.66, with confirmed zero
client overlap between train and test.

This is a large, meaningful gap (0.40 at Precision@20, 0.26 at Precision@50),
and it's a textbook case of what grouped validation is supposed to catch:
in the naive split, pages from the same client can appear in both train and
test. The model doesn't need to learn a generalizable pattern — it can
partly memorize client-specific quirks (a particular client's baseline
traffic level, content style, or update cadence) and then "recognize" that
client's other pages in the test set. That inflates the score without
representing real predictive skill on unseen clients.

The honest client-holdout number (0.60-0.66) is the trustworthy one. It's
also consistent with what I found in ML-08 using this same split design —
so this before/after comparison confirms that number wasn't a fluke, and
that the naive split would have overstated my model's real-world
performance by roughly 30-40 percentage points at these cutoffs. Any claim
about this model's performance should use the client-holdout numbers only.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Feature leakage audit for the features used in both splits above:
impressions_90d, sessions_90d, content_age_days, days_since_last_update,
avg_position, ctr, word_count.

- All seven features are observable at the decision point — none are
  computed from a future window relative to the label.
- trend_direction and trend_pct (the label source) are explicitly excluded
  from the feature list — confirmed by checking `features` above does not
  contain either.
- None of these features are product-computed decision flags (health_score,
  priority_score, action_type) per the lane guide's warning — they are all
  raw observable signals (impressions, sessions, position, CTR, age,
  staleness, word count).
- Group-level leakage: addressed directly by the client-holdout split in
  Section 2 — pages from the same client cannot appear in both train and
  test, which is the main leakage risk this dataset's structure creates.

No further leakage found beyond the split-level issue already demonstrated
and fixed in Section 2.

In [ ]:
label_source_cols = {'trend_direction', 'trend_pct'}
leaked = label_source_cols.intersection(set(features))
print(f"Label-derived columns found in feature set: {leaked if leaked else 'NONE — confirmed clean'}")

Label-derived columns found in feature set: NONE — confirmed clean


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Week 5) claim, as written in w05_model.ipynb:
"On this metric, this split, and this proxy label, the hand-written
baseline beats Random Forest at Precision@20 and Precision@50."

Audit of that claim: this claim already used the honest client-holdout
split from the start, so it does not need correcting for split leakage.
However, having now seen the naive-split numbers (0.60 -> 1.00 jump), I can
state more precisely WHY this matters: had I evaluated on a naive split
instead, I would have reported Precision@20 = 1.00, an obviously overstated
and unbelievable number that a careful reviewer should immediately question.

Rewritten claim, using safe language throughout:
"Under a client-holdout validation design — with zero client overlap
between train and test — the hand-written baseline rule observed a higher
Precision@20 and Precision@50 than a Random Forest model on this starter
dataset slice. This is a decision-support comparison on a proxy label
(same-window decline, not a true future outcome), not a causal or
production-grade benchmark. A naive random split on this same data
produces inflated, unreliable numbers (Precision@20 approaching 1.0), which
this audit confirms should never be reported as a real performance claim."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- Two paper findings named with methodology questions, framed constructively ✓
- Own model re-run under an honest split, with a real before/after
  comparison (naive 1.00/0.92 vs. honest 0.60/0.66) ✓
- Leakage audit completed: no label-derived columns in features, group-level
  leakage addressed via client-holdout ✓
- Claim rewritten using observed/directional/decision-support language,
  explicit about what would have gone wrong under a naive split ✓